# 🎛️ Deep Learning Studio · 3 of 5 — Pictures: Teach a Network to See 👕

**ISOM 839 · Prescriptive Analytics · Suffolk University · Prof. Hasan Arslan**

A clothing retailer gets thousands of returned items a day. Someone has to look at each one and sort it: shirt, sneaker, bag… You'll teach a network to do it from photos, decide **when to trust it**, and then learn the trick everyone uses today: **borrowing a network that already knows how to see.**

### 🧑‍🍳 The recipe (the same five steps in every Studio notebook)
| Step | Plain English |
|---|---|
| **1 · Numbers** | Turn the data into numbers |
| **2 · Shape** | Pick a network shape that fits the data |
| **3 · Train** | Let it practice: guess, measure the miss, nudge the knobs, repeat |
| **4 · Test** | Grade it on data it has never seen |
| **5 · Decide** | Use the prediction to make a business decision |

**Time:** about 25 minutes. 💡 *Runtime → Change runtime type → T4 GPU* makes training about 5× faster (it works without one too).

| Sign | Means |
|---|---|
| ▶️ **Run me** | press **Shift + Enter**, no typing needed |
| 🤔 **Bet first** | say your guess out loud *before* you run the next cell |
| ✅ **What you should see** | if your output is close to this, move on (small differences are normal) |
| ✏️ **Your turn** | change one thing and re-run |

---
## Setup

In [ ]:
# @title ▶️ Run me: setup (about 20 seconds)
import os, time, warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import keras
from keras import layers
from IPython.display import display, HTML
keras.utils.set_random_seed(839)

class Progress(keras.callbacks.Callback):
    # prints one short line every few rounds of practice
    def __init__(self, every=5):
        super().__init__(); self.every = every
    def on_epoch_end(self, epoch, logs=None):
        if epoch == 0 or (epoch + 1) % self.every == 0:
            print(f'round {epoch + 1:3d}:  ' + '   '.join(f'{k} {v:.3f}' for k, v in logs.items()))

def card(title, rows, color='#0a66c2'):
    body = ''.join(f"<tr><td style='padding:4px 18px 4px 0;color:#555'>{a}</td><td style='padding:4px 0;font-weight:700'>{b}</td></tr>" for a, b in rows)
    display(HTML(f"<div style='font:15px Arial;border-left:6px solid {color};padding:8px 16px;margin:6px 0;max-width:760px'>"
                 f"<div style='font-weight:700;font-size:16px;margin-bottom:4px'>{title}</div><table>{body}</table></div>"))

print('✅ Ready! Keras', keras.__version__)

**✅ What you should see:** `✅ Ready! Keras 3.…`

---
## Step 1 · Numbers — a picture is a grid of numbers

**Fashion-MNIST** is a classic set of 70,000 small photos of clothing from the retailer Zalando: 28 × 28 pixels, gray, 10 kinds of item. 60,000 are for practice; 10,000 are the exam.

In [ ]:
# @title ▶️ Run me: load the photos
(x_train, y_train), (x_exam, y_exam) = keras.datasets.fashion_mnist.load_data()
x_train = x_train[..., None].astype('float32') / 255        # brightness 0..255  ->  0..1
x_exam  = x_exam[..., None].astype('float32') / 255
names = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

fig, ax = plt.subplots(2, 10, figsize=(13, 3))
for k, a in enumerate(ax.ravel()):
    a.imshow(x_train[k, :, :, 0], cmap='gray'); a.set_title(names[y_train[k]], fontsize=9); a.axis('off')
plt.show()
print('Practice photos:', x_train.shape, '  Exam photos:', x_exam.shape)

**✅ What you should see:** 20 small photos with their names, and `Practice photos: (60000, 28, 28, 1)   Exam photos: (10000, 28, 28, 1)`.

In [ ]:
# @title ▶️ Run me: what the computer actually sees
photo = x_train[1, :, :, 0]
fig, ax = plt.subplots(1, 2, figsize=(12, 6))
ax[0].imshow(photo, cmap='gray'); ax[0].set_title('What you see'); ax[0].axis('off')
ax[1].imshow(photo, cmap='gray', alpha=0.25); ax[1].set_title('What the computer sees: 784 brightness numbers (0 = black, 9 = white)'); ax[1].axis('off')
for r in range(28):
    for c in range(28):
        ax[1].text(c, r, int(round(photo[r, c] * 9)), ha='center', va='center', fontsize=6)
plt.show()

**✅ What you should see:** a T-shirt on the left; on the right, the same T-shirt as a grid of digits. **Step 1 is already done for pictures: a photo *is* numbers.** (A color photo is three grids: red, green, blue.)

---
## Step 2 · Shape — first try: treat the photo like a spreadsheet row

Unroll the 28 × 28 grid into one long row of 784 numbers and use the `Dense` layers from Studio 1. The last layer has **10 neurons**, one per kind of item, and `softmax` turns them into 10 chances that add up to 100%.

In [ ]:
# @title ▶️ Run me: the "spreadsheet row" network (about 15 seconds)
flat_net = keras.Sequential([
    keras.Input(shape=(28, 28, 1)),
    layers.Flatten(),                              # unroll the grid into a row of 784 numbers
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax'),        # 10 chances, one per kind of item
])
flat_net.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
flat_net.fit(x_train, y_train, epochs=5, batch_size=128, validation_split=0.1, verbose=0, callbacks=[Progress(1)])
flat_accuracy = flat_net.evaluate(x_exam, y_exam, verbose=0)[1]
print(f'\nExam accuracy: {flat_accuracy:.1%}')

**✅ What you should see:** exam accuracy around **87%**. Not bad! But unrolling the photo throws away something important: **which pixels are neighbors**. A sleeve is a sleeve wherever it sits in the photo.

### A shape made for pictures: the convolutional network (CNN)

- **`Conv2D`** = a small **3 × 3 magnifying glass** that slides across the whole photo looking for one small pattern: an edge, a corner, a stripe. 32 magnifying glasses → 32 "pattern maps."
- **`MaxPooling2D`** = shrink each map to half size, keeping the strongest signals.
- Do it twice, and the second layer finds **patterns of patterns**: edges → sleeves and soles → shirts and sneakers.

The same magnifying glass is reused everywhere in the photo, so the network needs **fewer knobs** and learns faster.

In [ ]:
# @title ▶️ Run me: build the CNN
cnn = keras.Sequential([
    keras.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, 3, activation='relu', padding='same'),    # 32 magnifying glasses, 3 x 3 each
    layers.MaxPooling2D(),                                      # 28 x 28  ->  14 x 14
    layers.Conv2D(64, 3, activation='relu', padding='same'),    # 64 glasses looking at patterns of patterns
    layers.MaxPooling2D(),                                      # 14 x 14  ->  7 x 7
    layers.Flatten(),
    layers.Dropout(0.3),                                        # randomly silence 30% of signals while training: fights memorizing
    layers.Dense(10, activation='softmax'),
])
cnn.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
print('Knobs:  spreadsheet-row network', f'{flat_net.count_params():,}', '  CNN', f'{cnn.count_params():,}')

**✅ What you should see:** `Knobs:  spreadsheet-row network 101,770   CNN 50,186`. Half the knobs.

---
## Step 3 · Train

Same loop as always: guess, measure the miss, nudge the knobs. Eight rounds through the 60,000 photos.

In [ ]:
# @title ▶️ Run me: train the CNN (about 30 seconds on a GPU, 2 minutes without)
history = cnn.fit(x_train, y_train, epochs=8, batch_size=128, validation_split=0.1, verbose=0, callbacks=[Progress(1)])

**✅ What you should see:** 8 lines. `val_accuracy` (the practice exam) climbs from about 0.86 to about **0.91**.

In [ ]:
# @title ▶️ Run me: peek inside — what do the first magnifying glasses see?
first_layer = keras.Model(cnn.inputs, cnn.layers[0].output)
sneaker = x_exam[np.where(y_exam == 7)[0][0]][None]
maps = first_layer.predict(sneaker, verbose=0)[0]
fig, ax = plt.subplots(1, 9, figsize=(14, 2))
ax[0].imshow(sneaker[0, :, :, 0], cmap='gray'); ax[0].set_title('the photo', fontsize=9); ax[0].axis('off')
for k in range(8):
    ax[k + 1].imshow(maps[:, :, k], cmap='magma'); ax[k + 1].set_title(f'glass {k + 1}', fontsize=9); ax[k + 1].axis('off')
plt.show()

**✅ What you should see:** a sneaker, then 8 glowing versions of it. Each "glass" lights up for a different thing: one for the outline, one for the top edge, one for the bright fill… **Nobody programmed these. The network invented them** because they helped it reduce its mistakes.

---
## Step 4 · Test — 10,000 photos it has never seen

**🤔 Bet first:** which two items will the network confuse the most?

In [ ]:
# @title ▶️ Run me: grade the CNN and look at its mistakes
chances = cnn.predict(x_exam, verbose=0)                 # 10 chances for each exam photo
guess, confidence = chances.argmax(axis=1), chances.max(axis=1)
right = guess == y_exam
card('Exam accuracy', [('Spreadsheet-row network', f'{flat_accuracy:.1%}'), ('CNN', f'{right.mean():.1%}')])

mix = pd.crosstab(pd.Series(y_exam, name='really'), pd.Series(guess, name='said')).values.copy(); np.fill_diagonal(mix, 0)
print('Most common mix-ups:')
for i in np.argsort(-mix.ravel())[:3]:
    a, b = divmod(i, 10); print(f'   a {names[a]} called a {names[b]}: {mix[a, b]} times')

wrong = np.where(~right)[0][:8]
fig, ax = plt.subplots(1, 8, figsize=(14, 2.3))
for a, k in zip(ax, wrong):
    a.imshow(x_exam[k, :, :, 0], cmap='gray'); a.axis('off')
    a.set_title(f'said {names[guess[k]]} ({confidence[k]:.0%})\nwas {names[y_exam[k]]}', fontsize=8, color='#c0392b')
plt.show()

**✅ What you should see:** the CNN at about **90%**, a few points above the spreadsheet-row network. The mix-ups are **shirts, T-shirts, pullovers and coats**: the same ones a tired human would make from a tiny gray photo. Look at the 8 mistakes. Would you get them all right?

---
## Step 5 · Decide — when should the robot sort, and when a human?

91% is not 100%. But the network also tells us **how sure it is**. The returns center can use that:

| | |
|---|---|
| A human sorts an item | **\$0.40** |
| The robot sorts an item | free |
| A mis-sorted item (wrong shelf, wrong refund) | **\$3.00** |

**Rule:** the robot sorts an item only when it's at least *X%* sure. Otherwise a human looks. What's the best *X*?

In [ ]:
# @title ▶️ Run me: find the best "how sure" cutoff
HUMAN_COST, MISTAKE_COST = 0.40, 3.00                      # ✏️ change these later
rows = []
for cutoff in [0.00, 0.50, 0.70, 0.80, 0.90, 0.95, 0.99, 1.01]:
    robot = confidence >= cutoff                           # items the robot is sure enough about
    cost = ((~robot).sum() * HUMAN_COST + (robot & ~right).sum() * MISTAKE_COST) / len(y_exam)
    label = 'robot sorts everything' if cutoff == 0 else 'humans sort everything' if cutoff > 1 else f'robot if at least {cutoff:.0%} sure'
    rows.append({'rule': label, 'robot sorts': f'{robot.mean():.0%}', 'robot error rate': f'{(1 - right[robot].mean()) if robot.any() else 0:.1%}', 'cost per item': round(cost, 3)})
table = pd.DataFrame(rows); display(table)
best = table.loc[table['cost per item'].idxmin()]
print(f"Best rule: {best['rule']}  ->  ${best['cost per item']:.2f} per item instead of ${HUMAN_COST:.2f}.  On 1,000,000 returns a year that saves ${(HUMAN_COST - best['cost per item']) * 1_000_000:,.0f}.")

**✅ What you should see:** a table with 8 rules. Letting the robot sort everything costs about **\$0.30** an item (too many \$3 mistakes). The best rule is around **"robot if at least 90% sure"** (80% is nearly as good): the robot handles about **70%** of items, humans take the tricky rest, and the cost drops to about **\$0.15** an item, saving roughly **\$250,000** on a million returns.

That is how AI is really deployed: **not robot *or* human, but a cutoff that sends each item to the right one.**

### ✏️ Your turn
Mistakes get expensive: set `MISTAKE_COST = 20.00` and re-run. *(✅ The best cutoff jumps to 99%: the robot only sorts when it's nearly certain.)*

---
## 🚀 Floor 2 — borrow a network that already knows how to see

We had 60,000 labeled photos. Your company won't. A typical real project has **a few dozen photos per category**.

The modern answer: take a network that was **pretrained on millions of photos**, keep its "eyes," and train only one small new layer on top. This is called **transfer learning**.

The test: real color photos of 5 kinds of flowers, with only **40 practice photos per flower**.

In [ ]:
# @title ▶️ Run me: download 3,670 flower photos (about 20 seconds)
import glob
PHOTOS_PER_FLOWER = 40                                     # ✏️ try 10 later
archive = keras.utils.get_file('flower_photos.tgz', 'https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz', extract=True)
folder = os.path.dirname(glob.glob(os.path.join(os.path.dirname(archive), '**', 'daisy'), recursive=True)[0])
flowers = ['daisy', 'dandelion', 'roses', 'sunflowers', 'tulips']

def load(files):
    return np.stack([keras.utils.img_to_array(keras.utils.load_img(f, target_size=(224, 224))) for f in files])

rng = np.random.default_rng(839)
f_train, l_train, f_exam, l_exam = [], [], [], []
for k, flower in enumerate(flowers):
    files = sorted(glob.glob(os.path.join(folder, flower, '*.jpg'))); rng.shuffle(files)
    f_train.append(load(files[:PHOTOS_PER_FLOWER])); l_train += [k] * PHOTOS_PER_FLOWER
    f_exam.append(load(files[-100:]));               l_exam  += [k] * 100
f_train, f_exam, l_train, l_exam = np.concatenate(f_train), np.concatenate(f_exam), np.array(l_train), np.array(l_exam)

fig, ax = plt.subplots(1, 5, figsize=(13, 2.8))
for k, a in enumerate(ax):
    a.imshow(f_train[k * PHOTOS_PER_FLOWER].astype('uint8')); a.set_title(flowers[k]); a.axis('off')
plt.show()
print('Practice photos:', len(f_train), '  Exam photos:', len(f_exam))

**✅ What you should see:** five flower photos and `Practice photos: 200   Exam photos: 500`.

In [ ]:
# @title ▶️ Run me: from scratch vs borrowed eyes (about 1 minute)
# A) a CNN trained from scratch on only 200 photos
scratch = keras.Sequential([
    keras.Input(shape=(224, 224, 3)), layers.Rescaling(1 / 255),
    layers.Conv2D(16, 3, activation='relu'), layers.MaxPooling2D(),
    layers.Conv2D(32, 3, activation='relu'), layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation='relu'), layers.MaxPooling2D(),
    layers.GlobalAveragePooling2D(), layers.Dense(5, activation='softmax')])
scratch.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
scratch.fit(f_train, l_train, epochs=25, batch_size=32, verbose=0)
scratch_accuracy = scratch.evaluate(f_exam, l_exam, verbose=0)[1]

# B) borrowed eyes: a network pretrained on 1.3 million photos (ImageNet). We don't change it at all.
eyes = keras.applications.EfficientNetV2B0(include_top=False, weights='imagenet', pooling='avg')
seen_train = eyes.predict(f_train, verbose=0)              # each photo becomes 1,280 numbers: what the eyes noticed
seen_exam  = eyes.predict(f_exam, verbose=0)

head = keras.Sequential([keras.Input(shape=(1280,)), layers.Dense(5, activation='softmax')])   # the only part we train
head.compile(optimizer=keras.optimizers.Adam(0.003), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
head.fit(seen_train, l_train, epochs=40, batch_size=32, verbose=0)
borrowed_accuracy = head.evaluate(seen_exam, l_exam, verbose=0)[1]

card(f'Flowers, with only {PHOTOS_PER_FLOWER} practice photos each', [
    ('CNN trained from scratch', f'{scratch_accuracy:.0%}'),
    ('Borrowed eyes + one small trained layer', f'{borrowed_accuracy:.0%}'),
    ('Knobs we trained in the borrowed version', f'{head.count_params():,} (the eyes have {eyes.count_params():,} that we left alone)')], color='#2bb673')

**✅ What you should see:** from scratch about **50%**; borrowed eyes about **86%**. Same 200 photos. The pretrained network already knew edges, petals, textures and shapes from 1.3 million other photos; we only taught it five new words.

### ✏️ Your turn
Set `PHOTOS_PER_FLOWER = 10` in the download cell and re-run both cells. *(✅ With just 10 photos per flower, borrowed eyes still reach about 82%. From scratch falls to about 36%.)*

> **The frontier:** today's best "eyes" are *vision foundation models* such as Meta's **DINOv3**, trained on over a billion images with no human labels at all. And models that link pictures to words (the **CLIP** family) can sort photos into categories you simply *type*, with zero practice photos. Same idea as here, at a much larger scale.

---
## 🎯 What you just did
| Step | Here |
|---|---|
| **Numbers** | a photo is already a grid of brightness numbers |
| **Shape** | `Conv2D` + `MaxPooling2D`: small magnifying glasses that slide over the photo |
| **Train** | 8 rounds on 60,000 photos |
| **Test** | accuracy and the most common mix-ups on 10,000 new photos |
| **Decide** | a "how sure" cutoff that splits work between robot and human |
| **Floor 2** | borrowed pretrained eyes: great results from 40 photos per class |

**Next:** Studio 4 — **sound**. Spoiler: we turn it into a picture and reuse today's network. 🎵